# Kho-Sight — Auto-label remaining frames with your trained model

Use this AFTER your first training run (`train_khosight_colab.ipynb`). It runs your
fine-tuned role detector over the frames you haven't annotated yet and produces a
`frames + labels` zip you import into Roboflow — so the remaining annotation becomes
'skim and fix' instead of 'draw from scratch'.

**You need:**
- `khosight_roles_best.pt` (in your Drive at `MyDrive/khosight/` if you ran the training notebook)
- a zip of the UNLABELLED frames only (on your laptop: copy the frames you have not
  annotated into a folder, zip it, upload to Drive as `MyDrive/khosight/unlabelled.zip`).
  Tip: in Roboflow, filter Annotate -> 'Not Annotated' to see which filenames remain.

Runtime -> T4 GPU makes it fast (~10 min for 700 frames), but CPU works too.

In [ ]:
%pip -q install ultralytics

In [ ]:
# get weights + frames from Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/khosight/khosight_roles_best.pt /content/best.pt
!unzip -qo /content/drive/MyDrive/khosight/unlabelled.zip -d /content/unlabelled
import glob
# frames may be nested one level deep depending on how the zip was made
IMGS = sorted(glob.glob('/content/unlabelled/**/*.jpg', recursive=True) +
              glob.glob('/content/unlabelled/**/*.png', recursive=True))
print(len(IMGS), 'frames to label')

In [ ]:
# predict and write YOLO-format labels next to a copy of each image
import pathlib, shutil
from ultralytics import YOLO

CONF = 0.4   # raise for fewer-but-surer boxes, lower to catch more
out_img = pathlib.Path('/content/prelabelled/images'); out_img.mkdir(parents=True, exist_ok=True)
out_lbl = pathlib.Path('/content/prelabelled/labels'); out_lbl.mkdir(parents=True, exist_ok=True)

model = YOLO('/content/best.pt')
print('classes:', model.names)
for i in range(0, len(IMGS), 64):
    for res in model.predict(IMGS[i:i+64], conf=CONF, verbose=False):
        stem = pathlib.Path(res.path).stem
        h, w = res.orig_shape
        lines = []
        for box, cls in zip(res.boxes.xywhn.cpu().numpy(), res.boxes.cls.cpu().numpy()):
            cx, cy, bw, bh = box
            lines.append(f"{int(cls)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        (out_lbl / f'{stem}.txt').write_text('\n'.join(lines))
        shutil.copy(res.path, out_img / pathlib.Path(res.path).name)
    print(f'{min(i+64, len(IMGS))}/{len(IMGS)}')

# data.yaml so Roboflow maps class ids -> names correctly on import
names = [model.names[k] for k in sorted(model.names)]
pathlib.Path('/content/prelabelled/data.yaml').write_text(
    'train: images\nval: images\nnc: %d\nnames: %s\n' % (len(names), names))
print('done')

In [ ]:
# zip and download -> then in Roboflow: Upload Data -> drop the zip.
# Images arrive WITH annotations; correct them in Annotate as usual.
!cd /content && zip -qr prelabelled.zip prelabelled
!cp /content/prelabelled.zip /content/drive/MyDrive/khosight/prelabelled.zip
from google.colab import files
files.download('/content/prelabelled.zip')

## After importing into Roboflow
1. Skim each frame: fix wrong classes, add missed players, delete ghost boxes.
   Expect ~5-10 s per frame instead of 30-40 s.
2. When everything is annotated, generate a new dataset version (YOLOv11 export)
   and rerun `train_khosight_colab.ipynb` for the full-strength model.